## Pattern 16: Trie (Prefix Tree)

### 🧠 What is a Trie (Prefix Tree)?

A **Trie** (pronounced "try") is a **tree-like data structure** used to efficiently store and retrieve strings, especially when you need to search by **prefix**. Each path from root to a node represents a prefix of one or more stored words.

> **Think of it like a dictionary index** 📖 — Instead of scanning every word, you narrow down letter by letter. Looking for "cat"? Go to C → A → T. Done!

---

### 🔑 Key Characteristics

| Property | Description |
|----------|-------------|
| **Nodes** | Each node stores a `children` dictionary (char → child node) and an `is_end` flag |
| **Root** | The root node is empty — it represents the start of all words |
| **Edges** | Each edge represents a single character |
| **Shared Prefixes** | Words with common prefixes share the same path (saves space!) |

---

### 🔑 Core Idea — Visual Structure

Inserting the words **"cat"**, **"car"**, **"card"**, **"do"**, **"dog"**:

```
         (root)
        /      \
       c        d
       |        |
       a        o  ●    ← "do" ends here
      / \       |
     t●  r●     g●      ← "cat", "car", "dog" end here
         |
         d●              ← "card" ends here

● = is_end = True (a complete word ends at this node)
```

Notice how **"cat"** and **"car"** share the path `c → a`, and **"car"** and **"card"** share `c → a → r`. This prefix sharing is what makes Tries efficient!

---

### 📐 When to Use a Trie?

| Use Case | Why Trie Works |
|----------|----------------|
| **Autocomplete** | Find all words starting with a prefix |
| **Spell checker** | Check if a word exists in a dictionary |
| **IP routing** | Longest prefix matching |
| **Word games** | Validate words in Scrabble/Boggle |
| **Search engines** | Suggest queries as user types |

### 🪜 Step-by-Step Walkthrough

---

#### Operation 1: `insert("cat")`

Process each character and create nodes if they don't exist:

```
Step 1: At root → 'c' not in children → create node
         (root)
           |
           c

Step 2: At 'c' → 'a' not in children → create node
         (root)
           |
           c
           |
           a

Step 3: At 'a' → 't' not in children → create node → mark is_end = True
         (root)
           |
           c
           |
           a
           |
           t●         ← "cat" stored!
```

---

#### Operation 2: `insert("car")`

```
Step 1: At root → 'c' EXISTS → move to it (no new node!)
Step 2: At 'c' → 'a' EXISTS → move to it (no new node!)
Step 3: At 'a' → 'r' not in children → create node → mark is_end = True

         (root)
           |
           c
           |
           a
          / \
         t●  r●        ← "cat" and "car" share prefix "ca"
```

---

#### Operation 3: `search("cat")` → ✅ True

```
Step 1: At root → 'c' in children? ✅ → move to 'c'
Step 2: At 'c' → 'a' in children? ✅ → move to 'a'
Step 3: At 'a' → 't' in children? ✅ → move to 't'
Step 4: At 't' → is_end = True? ✅ → return True
```

---

#### Operation 4: `search("ca")` → ❌ False

```
Step 1: At root → 'c' in children? ✅ → move to 'c'
Step 2: At 'c' → 'a' in children? ✅ → move to 'a'
Step 3: At 'a' → is_end = True? ❌ → return False

"ca" is a PREFIX of stored words, but not a complete stored word itself.
```

---

#### Operation 5: `search("can")` → ❌ False

```
Step 1: At root → 'c' in children? ✅ → move to 'c'
Step 2: At 'c' → 'a' in children? ✅ → move to 'a'
Step 3: At 'a' → 'n' in children? ❌ → return False immediately
```

### 🐍 Code Walkthrough

#### `TrieNode` class
- **`children`** — A dictionary mapping characters to child `TrieNode`s
- **`is_end`** — Boolean flag, `True` if a complete word ends at this node

#### `Trie` class
- **`__init__()`** — Creates an empty Trie with a blank root node
- **`insert(word)`** — Walks through each character, creating new nodes as needed, marks the last node as `is_end = True`
- **`search(word)`** — Walks through each character, returns `False` if any character is missing, otherwise checks `is_end` at the final node

In [1]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False

In [3]:
class Trie:
    def __init__(self):
        self.root = TrieNode()
    
    def insert(self, word):
        node = self.root
        for char in word:
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
        node.is_end = True
    
    def search(self, word):
        node = self.root
        for char in word:
            if char not in node.children:
                return False
            node = node.children[char]
        return node.is_end


### ▶️ Example Usage

In [4]:
# --- Build a Trie and test operations ---
trie = Trie()

# Words to insert
words = ["cat", "car", "card", "care", "do", "dog", "done"]

print("=== Inserting Words ===\n")
for word in words:
    trie.insert(word)
    print(f'  insert("{word}") ✅')

# Search for words
print("\n=== Search Queries ===\n")
search_tests = ["cat", "car", "card", "ca", "can", "dog", "done", "dot"]
for word in search_tests:
    result = trie.search(word)
    symbol = "✅ Found" if result else "❌ Not Found"
    print(f'  search("{word}") → {symbol}')

# --- Bonus: starts_with (prefix search) ---
def starts_with(trie, prefix):
    """Check if any word in the trie starts with the given prefix"""
    node = trie.root
    for char in prefix:
        if char not in node.children:
            return False
        node = node.children[char]
    return True  # prefix exists, regardless of is_end

print("\n=== Prefix Queries ===\n")
prefixes = ["ca", "do", "car", "bat", "don"]
for prefix in prefixes:
    result = starts_with(trie, prefix)
    symbol = "✅ Prefix exists" if result else "❌ No words with this prefix"
    print(f'  starts_with("{prefix}") → {symbol}')

# --- Bonus: Get all words with a given prefix ---
def get_all_words(node, prefix, results):
    """Collect all words from a given node"""
    if node.is_end:
        results.append(prefix)
    for char, child in node.children.items():
        get_all_words(child, prefix + char, results)

def autocomplete(trie, prefix):
    """Return all words starting with the given prefix"""
    node = trie.root
    for char in prefix:
        if char not in node.children:
            return []
        node = node.children[char]
    results = []
    get_all_words(node, prefix, results)
    return results

print("\n=== Autocomplete ===\n")
for prefix in ["ca", "do", "car"]:
    suggestions = autocomplete(trie, prefix)
    print(f'  autocomplete("{prefix}") → {suggestions}')


=== Inserting Words ===

  insert("cat") ✅
  insert("car") ✅
  insert("card") ✅
  insert("care") ✅
  insert("do") ✅
  insert("dog") ✅
  insert("done") ✅

=== Search Queries ===

  search("cat") → ✅ Found
  search("car") → ✅ Found
  search("card") → ✅ Found
  search("ca") → ❌ Not Found
  search("can") → ❌ Not Found
  search("dog") → ✅ Found
  search("done") → ✅ Found
  search("dot") → ❌ Not Found

=== Prefix Queries ===

  starts_with("ca") → ✅ Prefix exists
  starts_with("do") → ✅ Prefix exists
  starts_with("car") → ✅ Prefix exists
  starts_with("bat") → ❌ No words with this prefix
  starts_with("don") → ✅ Prefix exists

=== Autocomplete ===

  autocomplete("ca") → ['cat', 'car', 'card', 'care']
  autocomplete("do") → ['do', 'dog', 'done']
  autocomplete("car") → ['car', 'card', 'care']


### ⏱️ Time & Space Complexity

#### Time Complexity

| Operation | Time Complexity | Explanation |
|-----------|----------------|-------------|
| `insert(word)` | **`O(m)`** | Traverse/create `m` nodes, where `m` = length of word |
| `search(word)` | **`O(m)`** | Traverse at most `m` nodes |
| `starts_with(prefix)` | **`O(m)`** | Same as search, just don't check `is_end` |

> **`m`** = length of the word/prefix being processed. Each operation is **independent of how many words** are stored — only the word length matters!

#### Comparison with Other Approaches

| Operation | Trie | Hash Set | Sorted Array |
|-----------|------|----------|--------------|
| Exact search | `O(m)` | `O(m)` | `O(m log n)` |
| Prefix search | `O(m)` ✅ | `O(n·m)` ❌ | `O(m log n)` |
| Insert | `O(m)` | `O(m)` | `O(n)` |
| Autocomplete | `O(m + k)` ✅ | `O(n·m)` ❌ | `O(m log n + k)` |

> `n` = total words, `m` = word length, `k` = number of matching results

#### Space Complexity: `O(N × C)`

| Component | Space |
|-----------|-------|
| Nodes | `O(N)` where N = total characters across all words (worst case) |
| Children dict per node | Up to `O(C)` where C = character set size (26 for lowercase English) |
| **Total** | **`O(N × C)`** worst case |

> In practice, shared prefixes **significantly reduce** space. E.g., storing "cat", "car", "card" uses only **6 nodes** instead of 10.

---

### 🌍 Classic Trie Problems

| Problem | Description |
|---------|-------------|
| **Implement Trie** | Basic insert, search, startsWith (LeetCode 208) |
| **Word Search II** | Find all words from dictionary in a 2D board (LeetCode 212) |
| **Longest Common Prefix** | Find longest prefix shared by all strings |
| **Replace Words** | Replace words in sentence with their root form |
| **Design Search Autocomplete** | Suggest top queries based on prefix |
| **Maximum XOR** | Find max XOR of two numbers using bitwise Trie |

---

### 💡 Key Takeaway

> **Trie = "A tree where each path spells a word, and shared prefixes share paths."**
> It makes prefix-based operations blazing fast at `O(m)`, regardless of how many words are stored — the only thing that matters is the **length of the word** you're looking up.

📋 Final Notebook Structure
| # | Cell Type | Content |
|---|-----------|---------|
| 1 | Markdown | **Pattern 16:** Trie (Prefix Tree) |
| 2 | Markdown | Intuition & Concept |
| 3 | Markdown | Step-by-Step Walkthrough |
| 4 | Markdown | Code Walkthrough |
| 5 | Code | `TrieNode` class |
| 6 | Code | `Trie` class |
| 7 | Markdown | Example Usage header |
| 8 | Code | Example + autocomplete |
| 9 | Markdown | Time/Space Complexity + Applications |